# 📊 Ind AS (2017) Taxonomy & Business Rules — Complete EDA & Architecture Guide

Is notebook me hum **`data/IND-AS taxonomy`** folder ko step-by-step explore (EDA) karenge taaki hume clear understanding ho sake:
1. **Folder Architecture**: Kaun-kaun se subdirectories aur XML files hain (`.xsd`, `*lab*.xml`, `*pre*.xml`, `*cal*.xml`, `*def*.xml`).
2. **Concepts (Elements)**: Balance Sheet, P&L, Notes ke tags kaise define hote hain (`in-ca` vs `ind-as` namespaces, `periodType`, `balance`, `type`).
3. **Presentation Linkbases (`*pre*.xml`)**: 66 ELRs ka parent-child presentation tree kaise banta hai.
4. **Calculation Linkbases (`*cal*.xml`)**: Arithmetic sums (weight +1 / -1) kaise define hote hain.
5. **Definition Linkbases (`*def*.xml`)**: Tables (Hypercubes), Dimensions (Axes), aur Members (`*Member`) kaise link hote hain.
6. **MCA Business Rules (`.xlsx`)**: Mandatory fields, conditional rules, aur validations kaise parse hoti hain.
7. **Final Output Schema**: Kaise ye sab combine hokar unified `ind_as_knowledge_base.json` banta hai.

--- 
## 1. Setup & Path Initialization

In [ ]:
# installing dependencies 

!pip install pandas


ModuleNotFoundError: No module named 'pandas'

In [14]:
from pathlib import Path
import os
import xml.etree.ElementTree as ET
import json
import pandas as pd
from python_calamine import CalamineWorkbook

# Resolve Paths
NOTEBOOK_DIR = Path(".").resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
INDAS_ROOT = PROJECT_ROOT / "data" / "IND-AS taxonomy"
TAXONOMY_DIR = INDAS_ROOT / "Taxonomy for IND-AS V1.2@31-03-2017" / "Ind AS taxonomy" / "Ind AS taxonomy"
RULES_EXCEL = INDAS_ROOT / "buisness_rules" / "BusinessRulesforIndASTaxonomyV1.2@-" / "Business Rules for IndAS Taxonomy V1.2.xlsx"
KB_JSON = PROJECT_ROOT / "knowledge_base" / "ind_as_knowledge_base.json"

print(f"Project Root: {PROJECT_ROOT}")
print(f"Taxonomy Dir Exists: {TAXONOMY_DIR.exists()}")
print(f"Rules Excel Exists: {RULES_EXCEL.exists()}")
print(f"Built KB JSON Exists: {KB_JSON.exists()}")

Project Root: /home/vanshsharma/projects/xbrl_autopilot
Taxonomy Dir Exists: True
Rules Excel Exists: True
Built KB JSON Exists: True


--- 
## 2. Directory Structure & File Counts
XBRL taxonomy me 5 type ke files hote hain:
- **`.xsd` (Schemas)**: Data dictionary (Concepts, Types, ELR Role definitions)
- **`*label*.xml` (Label Linkbase)**: English / Hindi names of each concept
- **`*pre*.xml` (Presentation Linkbase)**: Tree view hierarchy (Parent -> Children)
- **`*cal*.xml` (Calculation Linkbase)**: Math equations (A = B + C - D)
- **`*def*.xml` (Definition Linkbase)**: Multidimensional Tables & Schedules

In [15]:
xsds = list(TAXONOMY_DIR.rglob("*.xsd"))
labels = list(TAXONOMY_DIR.rglob("*lab*.xml"))
pres = list(TAXONOMY_DIR.rglob("*pre*.xml"))
cals = list(TAXONOMY_DIR.rglob("*cal*.xml"))
defs = list(TAXONOMY_DIR.rglob("*def*.xml"))

counts_df = pd.DataFrame([
    {"File Category": "Schemas (.xsd)", "Count": len(xsds), "Description": "Element, types, role definitions"},
    {"File Category": "Label Linkbases (*lab*.xml)", "Count": len(labels), "Description": "Human-readable labels (English, Hindi)"},
    {"File Category": "Presentation (*pre*.xml)", "Count": len(pres), "Description": "Hierarchical display trees (66 ELRs)"},
    {"File Category": "Calculation (*cal*.xml)", "Count": len(cals), "Description": "Arithmetic summation linkbases (39 ELRs)"},
    {"File Category": "Definition (*def*.xml)", "Count": len(defs), "Description": "Dimensional tables, axes, members (229 files)"},
])
counts_df

,File Category,Count,Description
0,Schemas (.xsd),8,"Element, types, role definitions"
1,Label Linkbases (*lab*.xml),2,"Human-readable labels (English, Hindi)"
2,Presentation (*pre*.xml),66,Hierarchical display trees (66 ELRs)
3,Calculation (*cal*.xml),39,Arithmetic summation linkbases (39 ELRs)
4,Definition (*def*.xml),229,"Dimensional tables, axes, members (229 files)"


--- 
## 3. The 8 XSD Schema Files (What Does Each Do?)
Dekhte hain ki 8 schema files kyu hain aur unka role kya hai:

In [16]:
xsd_summary = []
for x in sorted(xsds):
    rel = str(x.relative_to(TAXONOMY_DIR))
    size_kb = x.stat().st_size / 1024
    role = ""
    if "in-ca-ent" in rel:
        role = "Standalone Filing Entry Point (Main Schema)"
    elif "in-ci-ent" in rel:
        role = "Consolidated Filing Entry Point"
    elif "in-ca-2017" in rel:
        role = "MCA Companies Act Concepts (in-ca prefix)"
    elif "ind-as-2017" in rel:
        role = "ICAI Ind AS Accounting Standard Concepts (ind-as prefix)"
    elif "roles" in rel:
        role = "ELR Roles (Presentation role URIs & titles)"
    elif "types" in rel:
        role = "Custom Data Types (CIN, DIN, PAN, Monetary constraints)"
    elif "rol_for" in rel:
        role = "Formula linkbase roles"
    xsd_summary.append({"Relative Path": rel, "Size (KB)": f"{size_kb:.1f}", "Purpose": role})

pd.DataFrame(xsd_summary)

,Relative Path,Size (KB),Purpose
0,in-ca/in-ca-2017-03-31.xsd,224.1,MCA Companies Act Concepts (in-ca prefix)
1,in-ca/in-ca-roles-2017-03-31.xsd,11.1,ELR Roles (Presentation role URIs & titles)
2,in-ca/in-ca-types-2017-03-31.xsd,18.4,"Custom Data Types (CIN, DIN, PAN, Monetary con..."
3,in-ca-ent-2017-03-31.xsd,11.4,Standalone Filing Entry Point (Main Schema)
4,in-ci-ent-2017-03-31.xsd,72.7,Consolidated Filing Entry Point
5,ind-as/Formula/rol_for.xsd,2.7,Formula linkbase roles
6,ind-as/ind-as-2017-03-31.xsd,1299.5,ICAI Ind AS Accounting Standard Concepts (ind-...
7,ind-as/ind-as-roles-2017-03-31.xsd,77.8,ELR Roles (Presentation role URIs & titles)


--- 
## 4. Concepts Inspection: `in-ca` vs `ind-as`
XBRL me do primary namespaces hain:
1. **`in-ca`**: Indian Companies Act requirements (e.g. `CIN`, `PaidUpCapital`, `DetailsOfAuditors`)
2. **`ind-as`**: Ind AS Accounting Standards (e.g. `RevenueFromContractsWithCustomers`, `PropertyPlantAndEquipment`)

Har concept ke properties hoti hain:
- `name`: Local XML tag name
- `type`: `xbrli:monetaryItemType`, `nonnum:textBlockItemType`, `xbrli:stringItemType`
- `periodType`: `instant` (Balance Sheet - point in time) vs `duration` (P&L - over the year)
- `balance`: `debit` vs `credit`
- `abstract`: `True` (Heading/Group only, cannot hold amounts) vs `False` (Value row)

In [17]:
# Let's parse concepts from in-ca-2017-03-31.xsd
XS = "{http://www.w3.org/2001/XMLSchema}"
XBRLI = "{http://www.xbrl.org/2003/instance}"

in_ca_xsd = TAXONOMY_DIR / "in-ca" / "in-ca-2017-03-31.xsd"
tree = ET.parse(in_ca_xsd)
root = tree.getroot()

sample_concepts = []
for el in root.findall(f"{XS}element")[:10]:
    sample_concepts.append({
        "Name": el.attrib.get("name"),
        "Type": el.attrib.get("type", "").split(":")[-1],
        "Period": el.attrib.get(f"{XBRLI}periodType"),
        "Balance": el.attrib.get(f"{XBRLI}balance"),
        "IsAbstract": el.attrib.get("abstract") == "true",
    })

pd.DataFrame(sample_concepts)

,Name,Type,Period,Balance,IsAbstract
0,NameOfAssociateEntity,stringItemType,duration,NaN,False
1,DepositRepaymentReserveAccountWithBanks,monetaryItemType,instant,debit,False
2,DepositRepaymentReserveMember,domainItemType,duration,NaN,True
3,NumberOfSharesIssuedInPublicOffering,sharesItemType,duration,NaN,False
4,NumberOfSharesIssuedAsBonusShares,sharesItemType,duration,NaN,False
5,NumberOfSharesIssuedAsRights,sharesItemType,duration,NaN,False
6,NumberOfSharesIssuedInPrivatePlacementArisingO...,sharesItemType,duration,NaN,False
7,NumberOfSharesIssuedInOtherPrivatePlacement,sharesItemType,duration,NaN,False
8,NumberOfSharesIssuedAsPreferentialAllotmentAri...,sharesItemType,duration,NaN,False
9,NumberOfSharesIssuedAsOtherPreferentialAllotment,sharesItemType,duration,NaN,False


--- 
## 5. Labels Linkbase (`*label*.xml`)
XML element names (jaise `PropertyPlantAndEquipment`) ko human-friendly label (e.g. "Property, plant and equipment") se link karta hai.

In [18]:
LINK = "{http://www.xbrl.org/2003/linkbase}"
XLINK = "{http://www.w3.org/1999/xlink}"

in_ca_lab = TAXONOMY_DIR / "in-ca" / "in-ca-label-2017-03-31.xml"
tree = ET.parse(in_ca_lab)
root = tree.getroot()

sample_labels = []
for lbl in root.findall(f".//{LINK}label")[:10]:
    sample_labels.append({
        "ID": lbl.attrib.get(f"{XLINK}label"),
        "Lang": lbl.attrib.get("{http://www.w3.org/XML/1998/namespace}lang", "en"),
        "Role": lbl.attrib.get(f"{XLINK}role", "").split("/")[-1],
        "Text": (lbl.text or "").strip(),
    })

pd.DataFrame(sample_labels)

,ID,Lang,Role,Text
0,label_NameOfAssociateEntity,en,label,Name of associate entity
1,label_DepositRepaymentReserveAccountWithBanks,en,label,Deposit repayment reserve account with banks
2,label_DepositRepaymentReserveMember,en,label,Deposit repayment reserve [Member]
3,label_NumberOfSharesIssuedInPublicOffering,en,label,Number of shares issued in public offering
4,label_NumberOfSharesIssuedAsBonusShares,en,label,Number of shares issued as bonus shares
5,label_NumberOfSharesIssuedAsRights,en,label,Number of shares issued as rights
6,label_NumberOfSharesIssuedInPrivatePlacementAr...,en,label,Number of shares issued in private placement a...
7,label_NumberOfSharesIssuedInOtherPrivatePlacement,en,label,Number of shares issued in other private place...
8,label_NumberOfSharesIssuedAsPreferentialAllotm...,en,label,Number of shares issued as preferential allotm...
9,label_NumberOfSharesIssuedAsOtherPreferentialA...,en,label,Number of shares issued as other preferential ...


--- 
## 6. Presentation Linkbases (66 ELRs) — How the Tree is Built
Balance Sheet (`100100`), P&L (`100200`) ke presentation files me `presentationArc` hoti hai jo Parent -> Child order define karti hai.

In [19]:
bs_pre_file = TAXONOMY_DIR / "in-ca" / "Presentation" / "in-ca-pre-2017-03-31-100100.xml"
tree = ET.parse(bs_pre_file)
root = tree.getroot()

arcs = []
for arc in root.findall(f".//{LINK}presentationArc")[:12]:
    arcs.append({
        "Parent (From)": arc.attrib.get(f"{XLINK}from", "").split("_")[-1],
        "Child (To)": arc.attrib.get(f"{XLINK}to", "").split("_")[-1],
        "Order": float(arc.attrib.get("order", "1.0")),
        "Preferred Label": arc.attrib.get("preferredLabel", "").split("/")[-1] or "standard",
    })

pd.DataFrame(arcs)

FileNotFoundError: [Errno 2] No such file or directory: '/home/vanshsharma/projects/xbrl_autopilot/data/IND-AS taxonomy/Taxonomy for IND-AS V1.2@31-03-2017/Ind AS taxonomy/Ind AS taxonomy/in-ca/Presentation/in-ca-pre-2017-03-31-100100.xml'

--- 
## 7. Definition Linkbases (Tables, Dimensions, Members)
PPE Schedule, Share Capital table etc. dimensional schedules hoti hain jinki definition `*def*.xml` me hoti hai.

In [20]:
# Inspect PPE schedule definition
ppe_def_files = [f for f in defs if "400600" in f.name]
print(f"PPE definition files found: {len(ppe_def_files)}")

if ppe_def_files:
    tree = ET.parse(ppe_def_files[0])
    root = tree.getroot()
    def_arcs = []
    for arc in root.findall(f".//{LINK}definitionArc")[:10]:
        def_arcs.append({
            "Arcrole": arc.attrib.get(f"{XLINK}arcrole", "").split("/")[-1],
            "From": arc.attrib.get(f"{XLINK}from", "").split("_")[-1],
            "To": arc.attrib.get(f"{XLINK}to", "").split("_")[-1],
            "Order": float(arc.attrib.get("order", "1.0")),
        })
    display(pd.DataFrame(def_arcs))

PPE definition files found: 5


,Arcrole,From,To,Order
0,hypercube-dimension,DisclosureOfPropertyPlantAndEquipmentTable2NotAll,CarryingAmountAccumulatedDepreciationAndGrossC...,1.0
1,dimension-domain,CarryingAmountAccumulatedDepreciationAndGrossC...,GrossCarryingAmountMember,1.0


--- 
## 8. Business Rules Excel (`Business Rules for IndAS Taxonomy V1.2.xlsx`)
MCA ki official Excel workbook me 1,351 rules hain jo validation engine me use hoti hain.

In [21]:
wb = CalamineWorkbook.from_path(RULES_EXCEL)
print(f"Excel Sheets in MCA Business Rules: {wb.sheet_names}")

spec_sheet = next(s for s in wb.sheet_names if "specific" in s.lower())
rows = wb.get_sheet_by_name(spec_sheet).to_python()

rules_sample = []
for r in rows[2:12]:
    if len(r) > 5 and r[1] and r[5]:
        rules_sample.append({
            "Element Name": str(r[1]).strip(),
            "Rule Expression": str(r[5]).strip()[:80] + "...",
            "Severity / Status": str(r[7]).strip() if len(r) > 7 else "",
        })

pd.DataFrame(rules_sample)

Excel Sheets in MCA Business Rules: ['Applicable ELR', 'Specific rules for elements', 'Generic rules', 'Mandatory Line Items', 'Exempt parent member Dimension', 'Exempt Child Member Dimension', 'formulas Linkbase', 'Country Codes', 'Currency Code', 'Parent Child Exempt Calculation']


,Element Name,Rule Expression,Severity / Status
0,name,Business Rule...,Status
1,PropertyPlantAndEquipment,It is a mandatory field. Should be greater tha...,New Rules
2,OtherIntangibleAssets,It is a mandatory field. Should be greater tha...,New Rules


--- 
## 9. Inspecting the Unified JSON (`knowledge_base/ind_as_knowledge_base.json`)
Ye sabhi XML aur Excel files ko process karke humara builder script single high-speed JSON banata hai.
Dekhte hain ki is JSON ke top-level keys aur concept shapes kya hain:

In [22]:
with open(KB_JSON, "r", encoding="utf-8") as f:
    kb = json.load(f)

print("Top-level keys in Unified JSON:", list(kb.keys()))
print(f"Total Concepts: {len(kb['concepts'])}")
print(f"Total ELR Roles: {len(kb['roles'])}")
print(f"Total Presentation Roles: {len(kb['presentation'])}")
print(f"Total Calculation Roles: {len(kb['calculation'])}")
print(f"Total Definition Roles: {len(kb['definition'])}")

# Sample Concept in Unified JSON
sample_qn = "in-ca:EquityShareCapital"
print(f"\nSample Concept Structure ({sample_qn}):")
print(json.dumps(kb["concepts"].get(sample_qn), indent=2))

Top-level keys in Unified JSON: ['meta', 'namespaces', 'types', 'roles', 'concepts', 'presentation', 'calculation', 'definition', 'businessRules']
Total Concepts: 5647
Total ELR Roles: 246
Total Presentation Roles: 66
Total Calculation Roles: 39
Total Definition Roles: 229

Sample Concept Structure (in-ca:EquityShareCapital):
null


--- 
## 10. Summary & How It Connects to Our Software
- **`kb['concepts']`** $\rightarrow$ Powers Vector Search (`hybrid.py`), Auto-tagging suggestions, and Concept Details modal.
- **`kb['presentation']`** $\rightarrow$ Renders the right-hand taxonomy tree in `FormView.tsx` with identical Webtel hierarchy.
- **`kb['calculation']`** $\rightarrow$ Drives instant live reconciliation (`Difference = 0`).
- **`kb['definition']`** $\rightarrow$ Powers the Dedicated Table Tagging window (`[T]`) for PPE and dimensional schedules.
- **`kb['businessRules']`** $\rightarrow$ Runs the 200ms in-memory AST validation engine to give instant CA feedback without relying on laggy Java tools.

# MCA ALL ALL OVER THE WORLD  uses xml standard 2.1
- in which <link:linkbase 
        xmlns:link="http://www.xbrl.org/2003/linkbase"
        xmlns:xlink="http://www.w3.org/1999/xlink"            
        xmlns:xbrli="http://www.xbrl.org/2003/instance">  

these urls are the standard you can verify it in in-ca-label-2017-03-31.xml and this files in-ca-2017-03-31.xsd

In [5]:
from pathlib import Path
import xml.etree.ElementTree as ET

curr_dir = Path(".").resolve()

ROOT = curr_dir.parent

INDAS_DIR = ROOT / "data" / "IND-AS taxonomy"

# in-ca is for director cin reports and all
tax_dirs = [p for p in INDAS_DIR.rglob("in-ca") if p.is_dir()]
print("tax_dirs",tax_dirs)
if tax_dirs:
    TAXONOMY_DIR = tax_dirs[0].parent
    print(f"✓ Taxonomy Folder Found: {TAXONOMY_DIR.name}")
else:
    print("❌ Taxonomy folder nahi mila!")   


tax_dirs [PosixPath('/home/vanshsharma/projects/xbrl_autopilot/data/IND-AS taxonomy/Taxonomy for IND-AS V1.2@31-03-2017/Ind AS taxonomy/Ind AS taxonomy/in-ca')]
✓ Taxonomy Folder Found: Ind AS taxonomy


In [14]:
# inspecting xml headers
file_path=next(TAXONOMY_DIR.rglob("*lab*.xml"))
print(f"inspacting mca file name {file_path.name}")

tree = ET.parse(file_path)

for key, val in tree.getroot().attrib.items():
    print(f" {key} ---> {val}")


inspacting mca file name in-ca-label-2017-03-31.xml
 {http://www.w3.org/2001/XMLSchema-instance}schemaLocation ---> http://www.xbrl.org/2003/linkbase http://www.xbrl.org/2003/xbrl-linkbase-2003-12-31.xsd


 Financial statements me jo bhi row ya field hoti hai (jaise
  Equity Share Capital, Property Plant & Equipment, ya CIN),  
  use XBRL me Concept (ya Element) kaha jata hai.             
  
  Ye concepts MCA ki *.xsd files ke andar define hote hain:
                                                     

  • in-ca/in-ca-*.xsd → MCA Companies Act ke concepts (jaise
  CIN, PaidUpCapital)
  • ind-as/ind-as-*.xsd → ICAI Ind AS accounting standards ke
  concepts (jaise EquityShareCapital,
  PropertyPlantAndEquipment)         

 1. name: Tag ka naam (e.g. EquityShareCapital).
  2. type: Field me kya data aayega:                          
      • monetaryItemType: Rupee amounts (₹)
      • stringItemType: Text ya naam       
      • textBlockItemType: Poora narrative paragraph (Notes)
  3. periodType:                                              
      • instant: Point in time (As at 31st March → Balance
      Sheet item)                                             
      • duration: Poore saal ke liye (1st April to 31st March
      → Profit & Loss item)                                   
  4. balance:
      • debit: Asset ya Expense
      • credit: Liability, Equity, ya Income
  5. abstract:
      • True: Sirf Heading / Group Title hai (jaise "ASSETS"
      ya "Non-current assets"). Isme koi amount fill nahi ho
      sakti!
      • False: Real Value row hai jisme amount fill ki ja     
      sakti hai.

In [19]:
def clean_tag(tag):
    return tag.split("}")[-1] if "}" in tag else tag


def get_clean_attrib(element,attr_name):
    for k,v in element.attrib.items():
        if clean_tag(k) == attr_name:
            return v

    return None


concepts = {}
id_to_qname = {}
for xsd in sorted(TAXONOMY_DIR.rglob("*.xsd")):
    prefix = "in-ca" if "in-ca" in xsd.name else "ind-as"

    if not prefix:
        continue

    tree = ET.parse(xsd)

    for el in tree.getroot().iter():
        # if el is element fetch concepts

        if clean_tag(el.tag) == "element":
            name = el.attrib.get("name")
            if not name:
                continue

            qname = f"{prefix}:{name}"
            elem_id = el.attrib.get("id")

            concepts[qname] = {
                "qname": qname,
                "name": name,
                "prefix": prefix,
                "type": el.attrib.get("type","").split(":")[-1],
                "periodType": get_clean_attrib(el,"periodType"),
                "balance": get_clean_attrib(el,"balance")
                ,
                "abstract": el.attrib.get("abstract") == "true",
            }

            if elem_id:
                id_to_qname[elem_id] = qname
                id_to_qname[f"{xsd.name}#{elem_id}"] = qname

for lab_file in sorted(TAXONOMY_DIR.rglob("*lab*.xml")): 
    tree = ET.parse(lab_file)
    root = tree.getroot()

    loc_map = {}
    for el in root.iter():
        if clean_tag(el.tag) == "loc":
            loc_id = get_clean_attrib(el,"label")
            href = get_clean_attrib(el,"href") or ""
            frag = href.split("#")[-1]
            qn = id_to_qname.get(frag) or id_to_qname.get(frag.replace("_",":"))
            if qn:
                loc_map[loc_id] = qn

    # text map
    text_map = {}
    for el in root.iter():
        if clean_tag(el) == "label":
            lbl_id = get_clean_attrib(el,"label")
            lang = get_clean_attrib(el,"lang") or "en"
            text = (el.text or "").strip()
            text_map[lbl_id] = (text,lang)


    # labelArc
    for el in root.iter():
        if clean_tag(el) == "labelArc":
            from_loc = get_clean_attrib(el,"from")
            to_lbl = get_clean_attrib(el,"to")

            concept_qn = loc_map.get(from_loc)

            if concept_qn and to_lbl in text_map and concept_qn in concepts:
                txt, lang = text_map[to_lbl]
                if "labels" not in concepts[concept_qn]:
                    concepts[concept_qn]["labels"]= {}

                if lang == "hi":
                    concepts[concept_qn]["labels"]["label_hi"] = txt
                else:
                    concepts[concept_qn]["labels"]["label"] = txt


print(f"Total Concepts Parsed: {len(concepts)}")
print("--- Sample 1: Accounting Item (ind-as) ---")
print(concepts.get("ind-as:EquityShareCapital"))
print("\n--- Sample 2: Legal MCA Item (in-ca) ---")
print(concepts.get("in-ca:CorporateIdentityNumber"))


print("✓ Labels successfully merged into concepts!\n")    
  
sample1 = concepts.get("ind-as:EquityShareCapital")       
print("Sample 1 (ind-as):")
print(f"  QName: {sample1['qname']}")
print(f"  Human Label: {sample1.get('labels',             
{}).get('label')}")

sample2 = concepts.get("in-ca:CorporateIdentityNumber")
print("\nSample 2 (in-ca):") 
print(f"  QName: {sample2['qname']}")
print(f"  Human Label: {sample2.get('labels',    
{}).get('label')}")

Total Concepts Parsed: 5647
--- Sample 1: Accounting Item (ind-as) ---
{'qname': 'ind-as:EquityShareCapital', 'name': 'EquityShareCapital', 'prefix': 'ind-as', 'type': 'monetaryItemType', 'periodType': 'instant', 'balance': 'credit', 'abstract': False}

--- Sample 2: Legal MCA Item (in-ca) ---
{'qname': 'in-ca:CorporateIdentityNumber', 'name': 'CorporateIdentityNumber', 'prefix': 'in-ca', 'type': 'CINNumber', 'periodType': 'duration', 'balance': None, 'abstract': False}
✓ Labels successfully merged into concepts!

Sample 1 (ind-as):
  QName: ind-as:EquityShareCapital
  Human Label: None

Sample 2 (in-ca):
  QName: in-ca:CorporateIdentityNumber
  Human Label: None


In [20]:
presentations = {}

for pre_file in sorted(TAXONOMY_DIR.rglob("*.pre*.xml")):
    tree = ET.parse(pre_file)
    root  = tree.getroot()

    for link in root.iter():
        if clean_tag(link.tag) != "presentationLink":
            continue

        role_uri = get_clean_attrib(link,"role") or ""
        elr_code = role_uri.split("_")[-1] if "_" in role_uri else role_uri.split("/")[-1]

        # locators map (loc_id --> concept QName)
        locs = {}
        for el in link.iter():
            if clean_tag(el.tag) == "loc":
                l_id = get_clean_attrib(el,"label")
                frag = (get_clean_attrib(el,"href") or "").split("#")[-1]
                qn = id_to_qname.get(frag) or frag
                if qn:
                    locs[l_id] = qn


        # presentation se parent -- > child relationship extraction

        for arc in link.iter():
            if clean_tag(arc.tag) == "presentationoArc":
                from_parent = locs.get(get_clean_attrib(arc,"from"))
                to_child = locs.get(get_clean_attrib(arc,"to"))

                order = float(get_clean_attrib(arc,"order") or "1.0")

                if from_parent and to_child:
                    presentations.setdefault(elr_code,[]).append({
                        "from": from_parent,
                        "to": to_child,
                        "order": int(order) if order.is_integer() else order
                    })



print(f"Total Presentation {len(presentations)}")
print(f"Balance Sheet (110000) me total {len(presentations.get("110000",[]))} relationships \n")


print("--- Balance Sheet (110000) ke Pehle 6 Parent ->Child Arcs ---")
for arc in presentations.get("110000", [])[:6]:
    # Concept ka human label nikal kar display karte hain
    parent_lbl = concepts.get(arc["from"],      
    {}).get("labels", {}).get("label") or arc["from"] 
    child_lbl = concepts.get(arc["to"], {}).get("labels", 
    {}).get("label") or arc["to"]
    print(f"  Parent: {parent_lbl[:28]:28} ---> Child:
    (Order {arc['order']}) {child_lbl}") 


SyntaxError: unterminated f-string literal (detected at line 54) (2354953617.py, line 54)

In [ ]:
calculation = {}

# 1. Saari *cal*.xml files dynamically padho
for cal_file in sorted(TAXONOMY_DIR.rglob("*cal*.xml")):  
    tree = ET.parse(cal_file)
    root = tree.getroot()
    
    for link in root.iter():
        if clean_tag(link.tag) != "calculationLink":      
            continue
            
        role_uri = get_clean_attrib(link, "role") or ""   
        elr_code = role_uri.split("_")[-1] if "_" in      
        role_uri else role_uri.split("/")[-1]             
        
        # Step A: Locators map (loc_id -> concept QName)  
        locs = {}
        for el in link.iter():
            if clean_tag(el.tag) == "loc":
                l_id = get_clean_attrib(el, "label")      
                frag = (get_clean_attrib(el, "href") or   
                "").split("#")[-1]
                qn = id_to_qname.get(frag) or frag        
                if qn:
                    locs[l_id] = qn
                    
        # Step B: calculationArc se Math Equation nikalo 
        (weight +1 ya -1)
        for arc in link.iter():
            if clean_tag(arc.tag) == "calculationArc":
                total_concept =       
                locs.get(get_clean_attrib(arc, "from"))   
                component_concept =
                locs.get(get_clean_attrib(arc, "to"))     
                weight = float(get_clean_attrib(arc,
                "weight") or "1.0") 
                order = float(get_clean_attrib(arc,     
                "order") or "1.0") 
                
                if total_concept and component_concept: 
                    calculation.setdefault(elr_code, 
                    []).append({
                        "from": total_concept, 
                        "to": component_concept, 
                        "weight": int(weight) if         
                        weight.is_integer() else weight,
                        "order": int(order) if
                        order.is_integer() else order     
                    }) 

# 2. Output check karo 
print(f"✓ Total Calculation Statements/Notes (ELRs):      
{len(calculation)}") 
print(f"✓ Balance Sheet (110000) me       
{len(calculation.get('110000', []))} math equations
hain.\n")

print("--- Balance Sheet (110000) ke Pehle 5 Calculation  
Equations ---") 
for arc in calculation.get("110000", [])[:5]:
    total_lbl = concepts.get(arc["from"], 
    {}).get("labels", {}).get("label") or arc["from"]
    comp_lbl = concepts.get(arc["to"], {}).get("labels",
    {}).get("label") or arc["to"] 
    sign = "(+)" if arc["weight"] > 0 else "(-)"
    print(f"  Total: {total_lbl[:25]:25} {sign} Component:
    {comp_lbl}")

 Grid / Table form me hoti hain:                          
  
  1. Property, Plant & Equipment (PPE / Fixed Assets
  Schedule):                                                  
      • Rows: Land, Buildings, Plant & Machinery, Furniture,
      Vehicles...                                             
      • Columns: Opening Gross Block, Additions, Deductions,
      Depreciation, Net Block...                              
  2. Share Capital Schedule:    
      • Equity shares, Preference shares, Class A, Class B...
  3. Borrowings Schedule:                                     
      • Secured term loans, Debentures, Unsecured loans...
                                                              
  
  Inhe XBRL me Dimensional Tables (Hypercubes) kaha jata hai.
  Ind AS taxonomy me aisi 229 definition files (*def*.xml)    
  hain!         

In [21]:
definition = {} 
  
# 1. Saari *def*.xml files dynamically padho              
for def_file in sorted(TAXONOMY_DIR.rglob("*def*.xml")):  
    tree = ET.parse(def_file)
    root = tree.getroot()
    
    for link in root.iter():
        if clean_tag(link.tag) != "definitionLink":       
            continue
            
        role_uri = get_clean_attrib(link, "role") or ""   
        elr_code = role_uri.split("_")[-1] if "_" in      
        role_uri else role_uri.split("/")[-1]             
        
        # Step A: Locators map (loc_id -> concept QName)  
        locs = {}
        for el in link.iter():
            if clean_tag(el.tag) == "loc":
                l_id = get_clean_attrib(el, "label")      
                frag = (get_clean_attrib(el, "href") or   
                "").split("#")[-1]
                qn = id_to_qname.get(frag) or frag        
                if qn:
                    locs[l_id] = qn

        # Step B: definitionArc se Table, Axis aur Member 
        relationships nikalo
        for arc in link.iter():
            if clean_tag(arc.tag) == "definitionArc":     
                arcrole = (get_clean_attrib(arc,          
                "arcrole") or "").split("/")[-1]          
                from_concept =
                locs.get(get_clean_attrib(arc, "from"))   
                to_concept =
                locs.get(get_clean_attrib(arc, "to"))     
                order = float(get_clean_attrib(arc,       
                "order") or "1.0")

                if from_concept and to_concept:           
                    definition.setdefault(elr_code,       
                    []).append({
                        "arcrole": arcrole,
                        "from": from_concept,             
                        "to": to_concept,
                        "order": int(order) if            
                        order.is_integer() else order     
                    })

# 2. Output check karo
print(f"✓ Total Dimensional Schedule Files (ELRs):   
{len(definition)}")

# PPE Schedule (400600) ke members dekhte hain
ppe_arcs = definition.get("400600", [])
print(f"✓ PPE Schedule (400600) me total {len(ppe_arcs)} 
dimensional relationships hain.\n")

print("--- PPE Schedule (400600) ke Sample Table, Axis &
Members ---")
for arc in ppe_arcs[:8]:
    from_lbl = concepts.get(arc["from"], {}).get("labels",
    {}).get("label") or arc["from"]
    to_lbl = concepts.get(arc["to"], {}).get("labels",    
    {}).get("label") or arc["to"] 
    print(f"  [{arc['arcrole']:19}] {from_lbl[:26]:26}
    ---> {to_lbl}")
  ──────

SyntaxError: expected 'else' after 'if' expression (2348866817.py, line 13)

In [22]:
 
  XBRL me table 4 steps me banta hai:
  
    ┌────────────────────────────────────────────────────────┐
    │ 1. Table (Hypercube)                                   │
    │    e.g. PropertyPlantAndEquipmentTable                 │
    └───────────────────────────┬────────────────────────────┘
                                │ (hypercube-dimension)
                                ▼ 
    ┌────────────────────────────────────────────────────────┐
    │ 2. Axis (Dimension)                                    │
    │    e.g. ClassesOfPropertyPlantAndEquipmentAxis         │
    └───────────────────────────┬────────────────────────────┘
                                │ (dimension-domain)
                                ▼
    ┌────────────────────────────────────────────────────────┐
    │ 3. Domain / Parent Member                              │
    │    e.g. PropertyPlantAndEquipmentMember                │
    └───────────────────────────┬────────────────────────────┘
                                │ (domain-member) 
                                ▼
    ┌────────────────────────────────────────────────────────┐
    │ 4. Actual Members (Row Items)                          │
    │    • LandMember                                        │
    │    • BuildingsMember (Residential, Office, Factory)    │
    │    • PlantAndEquipmentMember                           │
    │    • FurnitureAndFixturesMember                        │
    └────────────────────────────────────────────────────────┘
  

SyntaxError: invalid character '┌' (U+250C) (159829927.py, line 3)

In [ ]:
from python_calamine import CalamineWorkbook
  
    # 1. Business Rules Excel file ko dynamically dhoondho 
    rules_file =
    next(INDAS_DATA_DIR.rglob("*Business*Rules*.xlsx"))       
    print(f"✓ Reading MCA Rules from: {rules_file.name}")
  
    wb_rules = CalamineWorkbook.from_path(rules_file)
    sheet_name = next(s for s in wb_rules.sheet_names if   
    "specific" in s.lower())
  
    rows = wb_rules.get_sheet_by_name(sheet_name).to_python()
  
    # Concept names se QName match karne ke liye lookup map
    name_to_qname = {c["name"].lower(): qn for qn, c in 
    concepts.items()}
  
    mandatory_concepts = []
    conditional_mandatory = [] 
    consolidated_mandatory = []
  
    # Row 3 se rules start hote hain
    for r in rows[3:]:
        if len(r) > 5 and r[1] and r[5]:
            elem_name = str(r[1]).strip()
            rule_text = str(r[5]).strip().lower()
            
            qn = name_to_qname.get(elem_name.lower())
            if not qn:
                continue
                
            if "mandatory in case of consolidated" in
            rule_text:
                if qn not in consolidated_mandatory:
                    consolidated_mandatory.append(qn)
            elif "mandatory field" in rule_text or "this is a
            mandatory field" in rule_text:
                if qn not in mandatory_concepts: 
                    mandatory_concepts.append(qn)
                    concepts[qn]["isMandatory"] = True
            elif "mandatory if" in rule_text:
                if qn not in conditional_mandatory:
                    conditional_mandatory.append(qn)
                    concepts[qn]["isMandatoryConditional"] =
                    True 
  
    # 2. Output check karo 
    print(f"✓ Globally Mandatory Concepts:   
         {len(mandatory_concepts)}")
    print(f"✓ Conditional Mandatory Concepts:    
      {len(conditional_mandatory)}")
    print(f"✓ Consolidated Mandatory Concepts:         
     {len(consolidated_mandatory)}\n")
  
    print("--- Sample Globally Mandatory Elements (Zero-fill 
    required if empty) ---") 
    for qn in mandatory_concepts[:5]:
        lbl = concepts.get(qn, {}).get("labels",           
        {}).get("label") or qn
        print(f"  • {qn:45} ({lbl})")

In [24]:
import json
from pathlib import Path 
import xml.etree.ElementTree as ET
from python_calamine import CalamineWorkbook

class IndASTaxonomyEngine:
    """Master Engine jo MCA IND-AS Taxonomy aur Business 
    Rules ko parse karke unified JSON banata hai.""" 
    
    def __init__(self, taxonomy_dir: Path, rules_excel_path: Path):
        self.tax_dir = taxonomy_dir
        self.rules_path = rules_excel_path
        self.concepts = {} 
        self.id_to_qname = {}
        self.presentation = {}
        self.calculation = {} 
        self.definition = {}
        self.rules = {}

    @staticmethod
    def clean_tag(tag):
        return tag.split("}")[-1] if "}" in tag else tag

    @staticmethod
    def get_clean_attrib(el, attr_name):
        for k, v in el.attrib.items(): 
            if IndASTaxonomyEngine.clean_tag(k) == attr_name:
                return v
        return None

    def parse_schemas(self): 
        """Step 1: Parse Concepts from all .xsd files"""
        print("  1/5 Parsing XSD Schemas (Concepts)...") 
        for xsd in sorted(self.tax_dir.rglob("*.xsd")):
            prefix = "in-ca" if "in-ca" in xsd.name else (
                "ind-as" if "ind-as" in xsd.name else None
            )
            if not prefix: 
                continue
            tree = ET.parse(xsd) 
            for el in tree.getroot().iter(): 
                if self.clean_tag(el.tag) == "element":
                    name = el.attrib.get("name")
                    if not name: 
                        continue
                    qname = f"{prefix}:{name}"
                    elem_id = el.attrib.get("id")
                    self.concepts[qname] = {
                        "qname": qname,
                        "name": name, 
                        "prefix": prefix,
                        "type": el.attrib.get("type", "").split(":")[-1],
                        "periodType": self.get_clean_attrib(el, "periodType"),
                        "balance": self.get_clean_attrib(el, "balance"),
                        "abstract": el.attrib.get("abstract") == "true",
                        "labels": {}
                    }
                    if elem_id:
                        self.id_to_qname[elem_id] = qname
                        self.id_to_qname[f"{xsd.name}#{elem_id}"] = qname 

    def parse_labels(self):
        """Step 2: Merge English and Hindi labels from *lab*.xml"""
        print("  2/5 Merging Labels Linkbases...")
        for lab in sorted(self.tax_dir.rglob("*lab*.xml")):          
            tree = ET.parse(lab)
            root = tree.getroot()
            loc_map = {}
            for el in root.iter():
                if self.clean_tag(el.tag) == "loc": 
                    l_id = self.get_clean_attrib(el, "label")
                    frag = (self.get_clean_attrib(el, "href") or "").split("#")[-1] 
                    qn = (
                        self.id_to_qname.get(frag) or   
                        self.id_to_qname.get(frag.replace("_", ":"))
                    )
                    if qn: 
                        loc_map[l_id] = qn 
            text_map = {} 
            for el in root.iter():
                if self.clean_tag(el.tag) == "label": 
                    l_id = self.get_clean_attrib(el, "label") 
                    lang = self.get_clean_attrib(el, "lang") or "en"
                    text_map[l_id] = ((el.text or "").strip(), lang)
            for el in root.iter():
                if self.clean_tag(el.tag) == "labelArc":
                    f = loc_map.get(self.get_clean_attrib(el, "from"))
                    t = self.get_clean_attrib(el, "to") 
                    if f and t in text_map and f in self.concepts:
                        txt, lang = text_map[t]
                        if lang == "hi": 
                            self.concepts[f]["labels"]["label_hi"] = txt 
                        else:
                            self.concepts[f]["labels"]["label"] = txt

    def parse_presentation(self):
        """Step 3: Parse Presentation Trees from *pre*.xml"""
        print("  3/5 Parsing Presentation Trees (ELRs)...")
        for pre in sorted(self.tax_dir.rglob("*pre*.xml")):          
            tree = ET.parse(pre)
            root = tree.getroot() 
            for link in root.iter():
                if self.clean_tag(link.tag) != "presentationLink": 
                    continue
                role = self.get_clean_attrib(link, "role") or "" 
                elr = role.split("_")[-1] if "_" in role else role.split("/")[-1]
                locs = {
                    self.get_clean_attrib(e, "label"): (
                        self.id_to_qname.get(
                            (self.get_clean_attrib(e, "href") or "").split("#")[-1]
                        ) or (self.get_clean_attrib(e, "href") or "").split("#")[-1]
                    ) for e in link.iter() if self.clean_tag(e.tag) == "loc"
                } 
                for arc in link.iter(): 
                    if self.clean_tag(arc.tag) == "presentationArc":
                        f = locs.get(self.get_clean_attrib(arc, "from"))
                        t = locs.get(self.get_clean_attrib(arc, "to"))
                        order = float(self.get_clean_attrib(arc, "order") or "1.0")
                        if f and t:
                            self.presentation.setdefault(elr, []).append({
                                "from": f,
                                "to": t, 
                                "order": order
                            }) 

    def parse_calculation(self):
        """Step 4: Parse Calculation Equations from *cal*.xml"""
        print("  4/5 Parsing Calculation Math Equations...")
        for cal in sorted(self.tax_dir.rglob("*cal*.xml")):          
            tree = ET.parse(cal) 
            root = tree.getroot() 
            for link in root.iter(): 
                if self.clean_tag(link.tag) != "calculationLink": 
                    continue
                role = self.get_clean_attrib(link, "role") or ""
                elr = role.split("_")[-1] if "_" in role else role.split("/")[-1]
                locs = {
                    self.get_clean_attrib(e, "label"): (
                        self.id_to_qname.get(
                            (self.get_clean_attrib(e, "href") or "").split("#")[-1]
                        ) or (self.get_clean_attrib(e, "href") or "").split("#")[-1]
                    ) for e in link.iter() if self.clean_tag(e.tag) == "loc"
                }
                for arc in link.iter(): 
                    if self.clean_tag(arc.tag) == "calculationArc":
                        f = locs.get(self.get_clean_attrib(arc, "from"))
                        t = locs.get(self.get_clean_attrib(arc, "to"))
                        w = float(self.get_clean_attrib(arc, "weight") or "1.0")
                        order = float(self.get_clean_attrib(arc, "order") or "1.0")
                        if f and t:
                            self.calculation.setdefault(elr, []).append({
                                "from": f,     
                                "to": t, 
                                "weight": w, 
                                "order": order
                            })

    def parse_business_rules(self):
        """Step 5: Parse MCA Business Rules from Excel"""
        print("  5/5 Parsing Business Rules from Excel...")
        wb = CalamineWorkbook.from_path(self.rules_path)
        sheet = next(s for s in wb.sheet_names if "specific" in s.lower())
        rows = wb.get_sheet_by_name(sheet).to_python() 
        name_map = {c["name"].lower(): qn for qn, c in self.concepts.items()}
        
        mandatory, conditional, consolidated = [], [], []
        for r in rows[3:]:
            if len(r) > 5 and r[1] and r[5]:
                name = str(r[1]).strip()
                rule = str(r[5]).strip().lower()
                qn = name_map.get(name.lower())
                if not qn: 
                    continue
                if "mandatory in case of consolidated" in rule: 
                    if qn not in consolidated:          
                        consolidated.append(qn) 
                elif "mandatory field" in rule or "this is a mandatory field" in rule:
                    if qn not in mandatory:           
                        mandatory.append(qn)
                elif "mandatory if" in rule:
                    if qn not in conditional:       
                        conditional.append(qn) 
                     
        self.rules = {
            "mandatoryConcepts": mandatory,
            "conditionalMandatory": conditional,
            "mandatoryConsolidated": consolidated
        }

    def build(self) -> dict:
        """Master builder executing all steps in sequence."""
        print("Starting Ind AS Taxonomy Engine Build...") 
        self.parse_schemas()
        self.parse_labels()
        self.parse_presentation()
        self.parse_calculation()
        self.parse_business_rules()
        
        final_kb = { 
            "meta": {
                "taxonomy": "ind_as", 
                "totalConcepts": len(self.concepts),
                "totalPresentationELRs": len(self.presentation),
                "totalCalculationELRs": len(self.calculation),
                "totalMandatoryConcepts": len(self.rules["mandatoryConcepts"]),     
            },
            "concepts": self.concepts,
            "presentation": self.presentation,            
            "calculation": self.calculation,
            "businessRules": self.rules,
        }
        print("✓ All 5 Steps Completed Successfully!")    
        return final_kb



NOTEBOOK_DIR = Path(".").resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
INDAS_ROOT = PROJECT_ROOT / "data" / "IND-AS taxonomy"
TAXONOMY_DIR = INDAS_ROOT / "Taxonomy for IND-AS V1.2@31-03-2017" / "Ind AS taxonomy" / "Ind AS taxonomy"
RULES_EXCEL = INDAS_ROOT / "buisness_rules" / "BusinessRulesforIndASTaxonomyV1.2@-" / "Business Rules for IndAS Taxonomy V1.2.xlsx"
KB_JSON = PROJECT_ROOT / "knowledge_base" / "ind_as_knowledge_base.json"

# Run Engine
engine = IndASTaxonomyEngine(taxonomy_dir=TAXONOMY_DIR, rules_excel_path=RULES_EXCEL)
complete_kb_json = engine.build()

print("\n--- Final Generated JSON Summary ---")           
print(json.dumps(complete_kb_json["meta"], indent=2))

Starting Ind AS Taxonomy Engine Build...
  1/5 Parsing XSD Schemas (Concepts)...
  2/5 Merging Labels Linkbases...
  3/5 Parsing Presentation Trees (ELRs)...
  4/5 Parsing Calculation Math Equations...
  5/5 Parsing Business Rules from Excel...
✓ All 5 Steps Completed Successfully!

--- Final Generated JSON Summary ---
{
  "taxonomy": "ind_as",
  "totalConcepts": 5647,
  "totalPresentationELRs": 60,
  "totalCalculationELRs": 37,
  "totalMandatoryConcepts": 220
}


In [25]:
import json
from pathlib import Path 
import xml.etree.ElementTree as ET
from python_calamine import CalamineWorkbook

class IndASTaxonomyEngine:
    """Master Engine jo MCA IND-AS Taxonomy aur Business 
    Rules ko parse karke unified JSON banata hai.""" 
    
    def __init__(self, taxonomy_dir: Path, rules_excel_path: Path):
        self.tax_dir = taxonomy_dir
        self.rules_path = rules_excel_path
        self.concepts = {} 
        self.id_to_qname = {}
        self.presentation = {}
        self.calculation = {} 
        self.definition = {}
        self.rules = {}

    @staticmethod
    def clean_tag(tag):
        return tag.split("}")[-1] if "}" in tag else tag

    @staticmethod
    def get_clean_attrib(el, attr_name):
        for k, v in el.attrib.items(): 
            if IndASTaxonomyEngine.clean_tag(k) == attr_name:
                return v
        return None

    def parse_schemas(self): 
        """Step 1: Parse Concepts from all .xsd files"""
        print("  1/6 Parsing XSD Schemas (Concepts)...") 
        for xsd in sorted(self.tax_dir.rglob("*.xsd")):
            prefix = "in-ca" if "in-ca" in xsd.name else (
                "ind-as" if "ind-as" in xsd.name else None
            )
            if not prefix: 
                continue
            tree = ET.parse(xsd) 
            for el in tree.getroot().iter(): 
                if self.clean_tag(el.tag) == "element":
                    name = el.attrib.get("name")
                    if not name: 
                        continue
                    qname = f"{prefix}:{name}"
                    elem_id = el.attrib.get("id")
                    self.concepts[qname] = {
                        "qname": qname,
                        "name": name, 
                        "prefix": prefix,
                        "type": el.attrib.get("type", "").split(":")[-1],
                        "periodType": self.get_clean_attrib(el, "periodType"),
                        "balance": self.get_clean_attrib(el, "balance"),
                        "abstract": el.attrib.get("abstract") == "true",
                        "labels": {}
                    }
                    if elem_id:
                        self.id_to_qname[elem_id] = qname
                        self.id_to_qname[f"{xsd.name}#{elem_id}"] = qname 

    def parse_labels(self):
        """Step 2: Merge English and Hindi labels from *lab*.xml"""
        print("  2/6 Merging Labels Linkbases...")
        for lab in sorted(self.tax_dir.rglob("*lab*.xml")):          
            tree = ET.parse(lab)
            root = tree.getroot()
            loc_map = {}
            for el in root.iter():
                if self.clean_tag(el.tag) == "loc": 
                    l_id = self.get_clean_attrib(el, "label")
                    frag = (self.get_clean_attrib(el, "href") or "").split("#")[-1] 
                    qn = (
                        self.id_to_qname.get(frag) or   
                        self.id_to_qname.get(frag.replace("_", ":"))
                    )
                    if qn: 
                        loc_map[l_id] = qn 
            text_map = {} 
            for el in root.iter():
                if self.clean_tag(el.tag) == "label": 
                    l_id = self.get_clean_attrib(el, "label") 
                    lang = self.get_clean_attrib(el, "lang") or "en"
                    text_map[l_id] = ((el.text or "").strip(), lang)
            for el in root.iter():
                if self.clean_tag(el.tag) == "labelArc":
                    f = loc_map.get(self.get_clean_attrib(el, "from"))
                    t = self.get_clean_attrib(el, "to") 
                    if f and t in text_map and f in self.concepts:
                        txt, lang = text_map[t]
                        if lang == "hi": 
                            self.concepts[f]["labels"]["label_hi"] = txt 
                        else:
                            self.concepts[f]["labels"]["label"] = txt

    def parse_presentation(self):
        """Step 3: Parse Presentation Trees from *pre*.xml"""
        print("  3/6 Parsing Presentation Trees (ELRs)...")
        for pre in sorted(self.tax_dir.rglob("*pre*.xml")):          
            tree = ET.parse(pre)
            root = tree.getroot() 
            for link in root.iter():
                if self.clean_tag(link.tag) != "presentationLink": 
                    continue
                role = self.get_clean_attrib(link, "role") or "" 
                elr = role.split("_")[-1] if "_" in role else role.split("/")[-1]
                locs = {
                    self.get_clean_attrib(e, "label"): (
                        self.id_to_qname.get(
                            (self.get_clean_attrib(e, "href") or "").split("#")[-1]
                        ) or (self.get_clean_attrib(e, "href") or "").split("#")[-1]
                    ) for e in link.iter() if self.clean_tag(e.tag) == "loc"
                } 
                for arc in link.iter(): 
                    if self.clean_tag(arc.tag) == "presentationArc":
                        f = locs.get(self.get_clean_attrib(arc, "from"))
                        t = locs.get(self.get_clean_attrib(arc, "to"))
                        order = float(self.get_clean_attrib(arc, "order") or "1.0")
                        if f and t:
                            self.presentation.setdefault(elr, []).append({
                                "from": f,
                                "to": t, 
                                "order": int(order) if order.is_integer() else order
                            }) 

    def parse_calculation(self):
        """Step 4: Parse Calculation Equations from *cal*.xml"""
        print("  4/6 Parsing Calculation Math Equations...")
        for cal in sorted(self.tax_dir.rglob("*cal*.xml")):          
            tree = ET.parse(cal) 
            root = tree.getroot() 
            for link in root.iter(): 
                if self.clean_tag(link.tag) != "calculationLink": 
                    continue
                role = self.get_clean_attrib(link, "role") or ""
                elr = role.split("_")[-1] if "_" in role else role.split("/")[-1]
                locs = {
                    self.get_clean_attrib(e, "label"): (
                        self.id_to_qname.get(
                            (self.get_clean_attrib(e, "href") or "").split("#")[-1]
                        ) or (self.get_clean_attrib(e, "href") or "").split("#")[-1]
                    ) for e in link.iter() if self.clean_tag(e.tag) == "loc"
                }
                for arc in link.iter(): 
                    if self.clean_tag(arc.tag) == "calculationArc":
                        f = locs.get(self.get_clean_attrib(arc, "from"))
                        t = locs.get(self.get_clean_attrib(arc, "to"))
                        w = float(self.get_clean_attrib(arc, "weight") or "1.0")
                        order = float(self.get_clean_attrib(arc, "order") or "1.0")
                        if f and t:
                            self.calculation.setdefault(elr, []).append({
                                "from": f,     
                                "to": t, 
                                "weight": int(w) if w.is_integer() else w, 
                                "order": int(order) if order.is_integer() else order
                            })

    def parse_definition(self):
        """Step 5: Parse Definition/Dimensional Linkbases from *def*.xml"""
        print("  5/6 Parsing Definition Dimensional Tables...")
        for def_file in sorted(self.tax_dir.rglob("*def*.xml")):
            tree = ET.parse(def_file)
            root = tree.getroot()
            for link in root.iter():
                if self.clean_tag(link.tag) != "definitionLink":
                    continue
                role_uri = self.get_clean_attrib(link, "role") or ""
                elr_code = role_uri.split("_")[-1] if "_" in role_uri else role_uri.split("/")[-1]
                locs = {
                    self.get_clean_attrib(e, "label"): (
                        self.id_to_qname.get(
                            (self.get_clean_attrib(e, "href") or "").split("#")[-1]
                        ) or (self.get_clean_attrib(e, "href") or "").split("#")[-1]
                    ) for e in link.iter() if self.clean_tag(e.tag) == "loc"
                }
                for arc in link.iter():
                    if self.clean_tag(arc.tag) == "definitionArc":
                        arcrole = (self.get_clean_attrib(arc, "arcrole") or "").split("/")[-1]
                        from_concept = locs.get(self.get_clean_attrib(arc, "from"))
                        to_concept = locs.get(self.get_clean_attrib(arc, "to"))
                        order = float(self.get_clean_attrib(arc, "order") or "1.0")
                        if from_concept and to_concept:
                            self.definition.setdefault(elr_code, []).append({
                                "arcrole": arcrole,
                                "from": from_concept,
                                "to": to_concept,
                                "order": int(order) if order.is_integer() else order
                            })

    def parse_business_rules(self):
        """Step 6: Parse MCA Business Rules from Excel"""
        print("  6/6 Parsing Business Rules from Excel...")
        wb = CalamineWorkbook.from_path(self.rules_path)
        sheet = next(s for s in wb.sheet_names if "specific" in s.lower())
        rows = wb.get_sheet_by_name(sheet).to_python() 
        name_map = {c["name"].lower(): qn for qn, c in self.concepts.items()}
        
        mandatory, conditional, consolidated = [], [], []
        for r in rows[3:]:
            if len(r) > 5 and r[1] and r[5]:
                name = str(r[1]).strip()
                rule = str(r[5]).strip().lower()
                qn = name_map.get(name.lower())
                if not qn: 
                    continue
                if "mandatory in case of consolidated" in rule: 
                    if qn not in consolidated:          
                        consolidated.append(qn) 
                elif "mandatory field" in rule or "this is a mandatory field" in rule:
                    if qn not in mandatory:           
                        mandatory.append(qn)
                        self.concepts[qn]["isMandatory"] = True
                elif "mandatory if" in rule:
                    if qn not in conditional:       
                        conditional.append(qn)
                        self.concepts[qn]["isMandatoryConditional"] = True
                     
        self.rules = {
            "mandatoryConcepts": mandatory,
            "conditionalMandatory": conditional,
            "mandatoryConsolidated": consolidated
        }

    def build(self) -> dict:
        """Master builder executing all steps in sequence."""
        print("Starting Ind AS Taxonomy Engine Build...") 
        self.parse_schemas()
        self.parse_labels()
        self.parse_presentation()
        self.parse_calculation()
        self.parse_definition()
        self.parse_business_rules()
        
        final_kb = { 
            "meta": {
                "taxonomy": "ind_as", 
                "totalConcepts": len(self.concepts),
                "totalPresentationELRs": len(self.presentation),
                "totalCalculationELRs": len(self.calculation),
                "totalDefinitionELRs": len(self.definition),
                "totalMandatoryConcepts": len(self.rules["mandatoryConcepts"]),     
            },
            "concepts": self.concepts,
            "presentation": self.presentation,            
            "calculation": self.calculation,
            "definition": self.definition,
            "businessRules": self.rules,
        }
        print("✓ All 6 Steps Completed Successfully!")    
        return final_kb

if __name__ == "__main__":
    NOTEBOOK_DIR = Path(".").resolve()
    PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
    INDAS_ROOT = PROJECT_ROOT / "data" / "IND-AS taxonomy"
    TAXONOMY_DIR = INDAS_ROOT / "Taxonomy for IND-AS V1.2@31-03-2017" / "Ind AS taxonomy" / "Ind AS taxonomy"
    RULES_EXCEL = INDAS_ROOT / "buisness_rules" / "BusinessRulesforIndASTaxonomyV1.2@-" / "Business Rules for IndAS Taxonomy V1.2.xlsx"
    
    # Destination path to save the final JSON knowledge base
    KB_JSON = PROJECT_ROOT / "knowledge_base" / "ind_as_knowledge_base.json"
    KB_JSON.parent.mkdir(parents=True, exist_ok=True)

    # Run Engine
    engine = IndASTaxonomyEngine(taxonomy_dir=TAXONOMY_DIR, rules_excel_path=RULES_EXCEL)
    complete_kb_json = engine.build()
  
    # Save to disk
    print(f"\nSaving Knowledge Base JSON to: {KB_JSON}")
    with open(KB_JSON, "w", encoding="utf-8") as f:
        json.dump(complete_kb_json, f, ensure_ascii=False, indent=2)
    print("✓ Knowledge Base successfully saved to disk!")

    print("\n--- Final Generated JSON Summary ---")           
    print(json.dumps(complete_kb_json["meta"], indent=2))

Starting Ind AS Taxonomy Engine Build...
  1/6 Parsing XSD Schemas (Concepts)...
  2/6 Merging Labels Linkbases...
  3/6 Parsing Presentation Trees (ELRs)...
  4/6 Parsing Calculation Math Equations...
  5/6 Parsing Definition Dimensional Tables...
  6/6 Parsing Business Rules from Excel...
✓ All 6 Steps Completed Successfully!

Saving Knowledge Base JSON to: /home/vanshsharma/projects/xbrl_autopilot/knowledge_base/ind_as_knowledge_base.json
✓ Knowledge Base successfully saved to disk!

--- Final Generated JSON Summary ---
{
  "taxonomy": "ind_as",
  "totalConcepts": 5647,
  "totalPresentationELRs": 60,
  "totalCalculationELRs": 37,
  "totalDefinitionELRs": 199,
  "totalMandatoryConcepts": 220
}


In [26]:
import json
from pathlib import Path
import xml.etree.ElementTree as ET
from python_calamine import CalamineWorkbook


class IndASTaxonomyEngine:
    """Master Engine jo MCA IND-AS Taxonomy aur Business Rules ko parse karke unified JSON banata hai."""

    def __init__(self, taxonomy_dir: Path, rules_excel_path: Path):
        self.tax_dir = taxonomy_dir
        self.rules_path = rules_excel_path
        self.concepts = {}
        self.id_to_qname = {}
        self.presentation = {}
        self.calculation = {}
        self.definition = {}
        self.rules = {}

    @staticmethod
    def clean_tag(tag: str) -> str:
        return tag.split("}")[-1] if "}" in tag else tag

    @staticmethod
    def get_clean_attrib(el: ET.Element, attr_name: str):
        for k, v in el.attrib.items():
            if IndASTaxonomyEngine.clean_tag(k) == attr_name:
                return v
        return None

    def parse_schemas(self):
        """Step 1: Parse Concepts from all .xsd files"""
        print("  1/6 Parsing XSD Schemas (Concepts)...")
        for xsd in sorted(self.tax_dir.rglob("*.xsd")):
            prefix = "in-ca" if "in-ca" in xsd.name else ("ind-as" if "ind-as" in xsd.name else None)
            if not prefix:
                continue
            tree = ET.parse(xsd)
            for el in tree.getroot().iter():
                if self.clean_tag(el.tag) == "element":
                    name = el.attrib.get("name")
                    if not name:
                        continue
                    qname = f"{prefix}:{name}"
                    elem_id = el.attrib.get("id")
                    self.concepts[qname] = {
                        "qname": qname,
                        "name": name,
                        "prefix": prefix,
                        "type": el.attrib.get("type", "").split(":")[-1],
                        "periodType": self.get_clean_attrib(el, "periodType"),
                        "balance": self.get_clean_attrib(el, "balance"),
                        "abstract": el.attrib.get("abstract") == "true",
                        "labels": {},
                    }
                    if elem_id:
                        self.id_to_qname[elem_id] = qname
                        self.id_to_qname[f"{xsd.name}#{elem_id}"] = qname

    def parse_labels(self):
        """Step 2: Merge English and Hindi labels from *lab*.xml"""
        print("  2/6 Merging Labels Linkbases...")
        for lab in sorted(self.tax_dir.rglob("*lab*.xml")):
            tree = ET.parse(lab)
            root = tree.getroot()
            loc_map = {}
            for el in root.iter():
                if self.clean_tag(el.tag) == "loc":
                    l_id = self.get_clean_attrib(el, "label")
                    frag = (self.get_clean_attrib(el, "href") or "").split("#")[-1]
                    qn = self.id_to_qname.get(frag) or self.id_to_qname.get(frag.replace("_", ":"))
                    if qn:
                        loc_map[l_id] = qn
            text_map = {}
            for el in root.iter():
                if self.clean_tag(el.tag) == "label":
                    l_id = self.get_clean_attrib(el, "label")
                    lang = self.get_clean_attrib(el, "lang") or "en"
                    text_map[l_id] = ((el.text or "").strip(), lang)
            for el in root.iter():
                if self.clean_tag(el.tag) == "labelArc":
                    f = loc_map.get(self.get_clean_attrib(el, "from"))
                    t = self.get_clean_attrib(el, "to")
                    if f and t in text_map and f in self.concepts:
                        txt, lang = text_map[t]
                        if lang == "hi":
                            self.concepts[f]["labels"]["label_hi"] = txt
                        else:
                            self.concepts[f]["labels"]["label"] = txt

    def parse_presentation(self):
        """Step 3: Parse Presentation Trees from *pre*.xml"""
        print("  3/6 Parsing Presentation Trees (ELRs)...")
        for pre in sorted(self.tax_dir.rglob("*pre*.xml")):
            tree = ET.parse(pre)
            root = tree.getroot()
            for link in root.iter():
                if self.clean_tag(link.tag) != "presentationLink":
                    continue
                role = self.get_clean_attrib(link, "role") or ""
                elr = role.split("_")[-1] if "_" in role else role.split("/")[-1]
                locs = {
                    self.get_clean_attrib(e, "label"): (
                        self.id_to_qname.get((self.get_clean_attrib(e, "href") or "").split("#")[-1])
                        or (self.get_clean_attrib(e, "href") or "").split("#")[-1]
                    )
                    for e in link.iter()
                    if self.clean_tag(e.tag) == "loc"
                }
                for arc in link.iter():
                    if self.clean_tag(arc.tag) == "presentationArc":
                        f = locs.get(self.get_clean_attrib(arc, "from"))
                        t = locs.get(self.get_clean_attrib(arc, "to"))
                        order = float(self.get_clean_attrib(arc, "order") or "1.0")
                        if f and t:
                            self.presentation.setdefault(elr, []).append({
                                "from": f,
                                "to": t,
                                "order": int(order) if order.is_integer() else order,
                            })

    def parse_calculation(self):
        """Step 4: Parse Calculation Equations from *cal*.xml"""
        print("  4/6 Parsing Calculation Math Equations...")
        for cal in sorted(self.tax_dir.rglob("*cal*.xml")):
            tree = ET.parse(cal)
            root = tree.getroot()
            for link in root.iter():
                if self.clean_tag(link.tag) != "calculationLink":
                    continue
                role = self.get_clean_attrib(link, "role") or ""
                elr = role.split("_")[-1] if "_" in role else role.split("/")[-1]
                locs = {
                    self.get_clean_attrib(e, "label"): (
                        self.id_to_qname.get((self.get_clean_attrib(e, "href") or "").split("#")[-1])
                        or (self.get_clean_attrib(e, "href") or "").split("#")[-1]
                    )
                    for e in link.iter()
                    if self.clean_tag(e.tag) == "loc"
                }
                for arc in link.iter():
                    if self.clean_tag(arc.tag) == "calculationArc":
                        f = locs.get(self.get_clean_attrib(arc, "from"))
                        t = locs.get(self.get_clean_attrib(arc, "to"))
                        w = float(self.get_clean_attrib(arc, "weight") or "1.0")
                        order = float(self.get_clean_attrib(arc, "order") or "1.0")
                        if f and t:
                            self.calculation.setdefault(elr, []).append({
                                "from": f,
                                "to": t,
                                "weight": int(w) if w.is_integer() else w,
                                "order": int(order) if order.is_integer() else order,
                            })

    def parse_definition(self):
        """Step 5: Parse Definition/Dimensional Linkbases from *def*.xml"""
        print("  5/6 Parsing Definition Dimensional Tables...")
        for def_file in sorted(self.tax_dir.rglob("*def*.xml")):
            tree = ET.parse(def_file)
            root = tree.getroot()
            for link in root.iter():
                if self.clean_tag(link.tag) != "definitionLink":
                    continue
                role_uri = self.get_clean_attrib(link, "role") or ""
                elr_code = role_uri.split("_")[-1] if "_" in role_uri else role_uri.split("/")[-1]
                locs = {
                    self.get_clean_attrib(e, "label"): (
                        self.id_to_qname.get((self.get_clean_attrib(e, "href") or "").split("#")[-1])
                        or (self.get_clean_attrib(e, "href") or "").split("#")[-1]
                    )
                    for e in link.iter()
                    if self.clean_tag(e.tag) == "loc"
                }
                for arc in link.iter():
                    if self.clean_tag(arc.tag) == "definitionArc":
                        arcrole = (self.get_clean_attrib(arc, "arcrole") or "").split("/")[-1]
                        from_concept = locs.get(self.get_clean_attrib(arc, "from"))
                        to_concept = locs.get(self.get_clean_attrib(arc, "to"))
                        order = float(self.get_clean_attrib(arc, "order") or "1.0")
                        if from_concept and to_concept:
                            self.definition.setdefault(elr_code, []).append({
                                "arcrole": arcrole,
                                "from": from_concept,
                                "to": to_concept,
                                "order": int(order) if order.is_integer() else order,
                            })

    def parse_business_rules(self):
        """Step 6: Parse MCA Business Rules from Excel"""
        print("  6/6 Parsing Business Rules from Excel...")
        wb = CalamineWorkbook.from_path(self.rules_path)
        sheet = next(s for s in wb.sheet_names if "specific" in s.lower())
        rows = wb.get_sheet_by_name(sheet).to_python()
        name_map = {c["name"].lower(): qn for qn, c in self.concepts.items()}

        mandatory, conditional, consolidated = [], [], []
        for r in rows[3:]:
            if len(r) > 5 and r[1] and r[5]:
                name = str(r[1]).strip()
                rule = str(r[5]).strip().lower()
                qn = name_map.get(name.lower())
                if not qn:
                    continue
                if "mandatory in case of consolidated" in rule:
                    if qn not in consolidated:
                        consolidated.append(qn)
                        self.concepts[qn]["isMandatoryConsolidated"] = True
                elif "mandatory field" in rule or "this is a mandatory field" in rule:
                    if qn not in mandatory:
                        mandatory.append(qn)
                        self.concepts[qn]["isMandatory"] = True
                elif "mandatory if" in rule:
                    if qn not in conditional:
                        conditional.append(qn)
                        self.concepts[qn]["isMandatoryConditional"] = True

        self.rules = {
            "mandatoryConcepts": mandatory,
            "conditionalMandatory": conditional,
            "mandatoryConsolidated": consolidated,
        }

    def build(self) -> dict:
        """Master builder executing all steps in sequence."""
        print("Starting Ind AS Taxonomy Engine Build...")
        self.parse_schemas()
        self.parse_labels()
        self.parse_presentation()
        self.parse_calculation()
        self.parse_definition()
        self.parse_business_rules()

        final_kb = {
            "meta": {
                "taxonomy": "ind_as",
                "totalConcepts": len(self.concepts),
                "totalPresentationELRs": len(self.presentation),
                "totalCalculationELRs": len(self.calculation),
                "totalDefinitionELRs": len(self.definition),
                "totalMandatoryConcepts": len(self.rules["mandatoryConcepts"]),
                "totalConditionalMandatory": len(self.rules["conditionalMandatory"]),
                "totalConsolidatedMandatory": len(self.rules["mandatoryConsolidated"]),
            },
            "concepts": self.concepts,
            "presentation": self.presentation,
            "calculation": self.calculation,
            "definition": self.definition,
            "businessRules": self.rules,
        }
        print("✓ All 6 Steps Completed Successfully!")
        return final_kb


if __name__ == "__main__":
    NOTEBOOK_DIR = Path(".").resolve()
    PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
    INDAS_ROOT = PROJECT_ROOT / "data" / "IND-AS taxonomy"
    TAXONOMY_DIR = INDAS_ROOT / "Taxonomy for IND-AS V1.2@31-03-2017" / "Ind AS taxonomy" / "Ind AS taxonomy"
    RULES_EXCEL = INDAS_ROOT / "buisness_rules" / "BusinessRulesforIndASTaxonomyV1.2@-" / "Business Rules for IndAS Taxonomy V1.2.xlsx"

    # Destination path to save the final JSON knowledge base
    KB_JSON = PROJECT_ROOT / "knowledge_base" / "ind_as_knowledge_base.json"
    KB_JSON.parent.mkdir(parents=True, exist_ok=True)

    # Run Engine
    engine = IndASTaxonomyEngine(taxonomy_dir=TAXONOMY_DIR, rules_excel_path=RULES_EXCEL)
    complete_kb_json = engine.build()

    # Save to disk
    print(f"\nSaving Knowledge Base JSON to: {KB_JSON}")
    with open(KB_JSON, "w", encoding="utf-8") as f:
        json.dump(complete_kb_json, f, ensure_ascii=False, indent=2)
    print("✓ Knowledge Base successfully saved to disk!")

    print("\n--- Final Generated JSON Summary ---")
    print(json.dumps(complete_kb_json["meta"], indent=2))


Starting Ind AS Taxonomy Engine Build...
  1/6 Parsing XSD Schemas (Concepts)...
  2/6 Merging Labels Linkbases...
  3/6 Parsing Presentation Trees (ELRs)...
  4/6 Parsing Calculation Math Equations...
  5/6 Parsing Definition Dimensional Tables...
  6/6 Parsing Business Rules from Excel...
✓ All 6 Steps Completed Successfully!

Saving Knowledge Base JSON to: /home/vanshsharma/projects/xbrl_autopilot/knowledge_base/ind_as_knowledge_base.json
✓ Knowledge Base successfully saved to disk!

--- Final Generated JSON Summary ---
{
  "taxonomy": "ind_as",
  "totalConcepts": 5647,
  "totalPresentationELRs": 60,
  "totalCalculationELRs": 37,
  "totalDefinitionELRs": 199,
  "totalMandatoryConcepts": 220,
  "totalConditionalMandatory": 173,
  "totalConsolidatedMandatory": 1
}
